In [7]:
import os
import json
import time
import requests
from dotenv import load_dotenv

# ✅ Load API key
load_dotenv()
API_KEY = os.getenv("GOOGLE_MAP_API_KEY")
if not API_KEY:
    raise ValueError("❌ GOOGLE_MAP_API_KEY not found in environment.")

# --- CONFIG ---
SOURCE_URL = "https://raw.githubusercontent.com/ThathsaraniPathirana/LLM-project/refs/heads/main/Store/all_stores_sweden_flat.json"
OUTPUT_FILE = "ratings_stores.json"
BATCH_SIZE = 50
SLEEP_SEC = 1

# --- Utility functions ---
def extract_value(v):
    """Handles string, dict(@value), or None safely."""
    if isinstance(v, dict):
        return v.get("@value")
    elif isinstance(v, str):
        return v.strip()
    else:
        return None

def build_query(name, alt, street, city):
    """Combine name, alt, street, city into one Google textQuery."""
    parts = [name, alt, street, city, "Sweden"]
    return ", ".join(str(p).strip() for p in parts if p)

# --- Load source JSON ---
print("📂 Fetching store dataset...")
r = requests.get(SOURCE_URL, timeout=30)
r.raise_for_status()
data = r.json()
print(f"✅ Loaded {len(data)} store entries from VisitSweden dataset.")

# --- Resume progress if exists ---
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        results = json.load(f)
    processed = {r["name"] for r in results}
    print(f"🔁 Resuming — already processed {len(processed)} stores.")
else:
    results = []
    processed = set()

# --- Google Places API setup ---
url = "https://places.googleapis.com/v1/places:searchText"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": (
        "places.displayName,"
        "places.rating,"
        "places.userRatingCount,"
        "places.formattedAddress,"
        "places.googleMapsUri"
    ),
}

count = len(results)
for record in data:
    name = record.get("name")
    if not name or name in processed:
        continue

    alt = record.get("alternate_name")
    street = extract_value(record.get("street"))
    city = extract_value(record.get("city"))

    query = build_query(name, alt, street, city)
    payload = {"textQuery": query}

    try:
        res = requests.post(url, headers=headers, json=payload, timeout=20)
        data = res.json()

        if "error" in data:
            print(f"⚠️ API error for {name}: {data['error'].get('message')}")
            continue

        places = data.get("places", [])
        if not places:
            print(f"❌ No results for {name}")
            continue

        place = places[0]
        entry = {
            "name": name,
            "alternate_name": alt,
            "rating": place.get("rating"),
            "userRatingCount": place.get("userRatingCount"),
            "formattedAddress": place.get("formattedAddress"),
            "googleMapsUri": place.get("googleMapsUri"),
        }

        results.append(entry)
        processed.add(name)
        count += 1

        print(f"✅ [{count}] {name} → ⭐ {entry['rating']} ({entry['userRatingCount']})")

        # Save periodically
        if count % BATCH_SIZE == 0:
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            print(f"💾 Saved progress — {count} processed.")
            time.sleep(2)

        time.sleep(SLEEP_SEC)

    except Exception as e:
        print(f"⚠️ Error for {name}: {e}")
        time.sleep(3)
        continue

# --- Final save ---
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n🎯 Done! Saved {len(results)} entries to {OUTPUT_FILE}")


📂 Fetching store dataset...
✅ Loaded 764 store entries from VisitSweden dataset.
✅ [1] Härvan → ⭐ 3.9 (930)
✅ [2] Kreativboden → ⭐ 4.6 (16)
✅ [3] Gnappes butik - nytt & gammalt → ⭐ None (None)
✅ [4] Häggs Bildemontering - Ilsbo → ⭐ 4.4 (204)
✅ [5] Vänskapa Design & Hantverk → ⭐ 5 (1)
✅ [6] Antikvariat Furioso → ⭐ 4.9 (10)
✅ [7] Skärså Rökeri → ⭐ 4.5 (312)
✅ [8] InfoPoint Hällåsen Aquarena → ⭐ 4.2 (314)
✅ [9] InfoPoint Stenö havsbad och camping → ⭐ 4 (1211)
✅ [10] InfoPoint Glasstantens lanthandel (sommar) → ⭐ 5 (1)
✅ [11] InfoPoint Café Rådis i Söderhamn → ⭐ 4.5 (314)
✅ [12] InfoPoint Världsarvsgården Erik-Anders (sommar) → ⭐ 4.1 (27)
✅ [13] InfoPoint First Hotel Statt → ⭐ 3.5 (488)
✅ [14] InfoPoint Best Western hotell → ⭐ 3.8 (669)
✅ [15] InfoPoint Söderhamns kundtjänst → ⭐ 4 (7)
✅ [16] Glasstantens lanthandel → ⭐ 5 (1)
✅ [17] Guldfynd → ⭐ 4.2 (19)
❌ No results for House of Helsingland
✅ [18] Gårdshandeln i Högbo → ⭐ 4.2 (36)
✅ [19] Slöjd i sjöboden → ⭐ 4.6 (24)
✅ [20] Himmel & Jord K